In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
!pip install -q langchain langchain-core langchain-community langchain-groq

In [3]:
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage

In [4]:
from langchain_groq import ChatGroq
from kaggle_secrets import UserSecretsClient
secret_label = "GROQ_API_KEY"
api_key = UserSecretsClient().get_secret(secret_label)

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    api_key=api_key
)

# Create Tool

In [5]:
@tool
def generate_hypotenuse(a:int, b:int) -> float:
    """given two numbers(base and height) this tool generates the hypotenuse"""
    return ((a**2)+(b**2))**0.5

In [6]:
print(generate_hypotenuse.invoke({'a':3,'b':4}))

5.0


# Tool Binding
### bind tool with llm

In [7]:
llm_with_tools = llm.bind_tools([generate_hypotenuse])
llm_with_tools

RunnableBinding(bound=ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 32768, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x783a0c4d9a30>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x783a0b45a150>, model_name='llama-3.3-70b-versatile', model_kwargs={}, groq_api_key=SecretStr('**********')), kwargs={'tools': [{'type': 'function', 'function': {'name': 'generate_hypotenuse', 'description': 'given two numbers(base and height) this tool generates the hypotenuse', 'parameters': {'properties': {'a': {'type': 'integer'}, 'b': {'type': 'integer'}}, 'required': ['a', 'b'], 'type': 'object'}}}]}, config={}, config_factories=[])

In [8]:
llm_with_tools.invoke("What is the fullform of HTTP?")

AIMessage(content='HTTP stands for HyperText Transfer Protocol.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 9, 'prompt_tokens': 250, 'total_tokens': 259, 'completion_time': 0.017968353, 'completion_tokens_details': None, 'prompt_time': 0.051440247, 'prompt_tokens_details': None, 'queue_time': 0.268255478, 'total_time': 0.0694086}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_45180df409', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019d2343-6235-79e2-8ec7-ba5aee9e38c8-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 250, 'output_tokens': 9, 'total_tokens': 259})

In [9]:
query = HumanMessage('a right angled triangle has base 5 and height 12. what is the hypotenuse?')

In [10]:
messages = [query]
messages  #HumanMessage

[HumanMessage(content='a right angled triangle has base 5 and height 12. what is the hypotenuse?', additional_kwargs={}, response_metadata={})]

In [11]:
result = llm_with_tools.invoke(messages)
messages.append(result)
messages  #HumanMessage. AIMessage

[HumanMessage(content='a right angled triangle has base 5 and height 12. what is the hypotenuse?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '649k8az0k', 'function': {'arguments': '{"a":5,"b":12}', 'name': 'generate_hypotenuse'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 23, 'prompt_tokens': 262, 'total_tokens': 285, 'completion_time': 0.05138385, 'completion_tokens_details': None, 'prompt_time': 0.014903174, 'prompt_tokens_details': None, 'queue_time': 0.105488034, 'total_time': 0.066287024}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019d2343-63fc-70d2-8fe7-47f4570cc883-0', tool_calls=[{'name': 'generate_hypotenuse', 'args': {'a': 5, 'b': 12}, 'id': '649k8az0k', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_

In [12]:
result.tool_calls[0]

{'name': 'generate_hypotenuse',
 'args': {'a': 5, 'b': 12},
 'id': '649k8az0k',
 'type': 'tool_call'}

In [13]:
tool_result = generate_hypotenuse.invoke(result.tool_calls[0])
tool_result

ToolMessage(content='13.0', name='generate_hypotenuse', tool_call_id='649k8az0k')

In [14]:
messages.append(tool_result)
messages  #HumanMessage, AIMessage, ToolMessage

[HumanMessage(content='a right angled triangle has base 5 and height 12. what is the hypotenuse?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '649k8az0k', 'function': {'arguments': '{"a":5,"b":12}', 'name': 'generate_hypotenuse'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 23, 'prompt_tokens': 262, 'total_tokens': 285, 'completion_time': 0.05138385, 'completion_tokens_details': None, 'prompt_time': 0.014903174, 'prompt_tokens_details': None, 'queue_time': 0.105488034, 'total_time': 0.066287024}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019d2343-63fc-70d2-8fe7-47f4570cc883-0', tool_calls=[{'name': 'generate_hypotenuse', 'args': {'a': 5, 'b': 12}, 'id': '649k8az0k', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_

In [15]:
llm_with_tools.invoke(messages).content

'The hypotenuse of the right-angled triangle is 13.0 units.'

# Expense Tracker & Analyzer

In [16]:
import json

In [32]:
@tool
def tracker(expense: str) -> list:
    """Extracts expenses from text and returns a list of expense dictionaries."""

    prompt = f"""
    Extract all expenses from the text below.
    
    Text: "{expense}"
    
    Return ONLY a valid JSON list of dictionaries.
    Do NOT include any explanation, text, or markdown formatting like ```json.
    
    Example:
    [
        {{"amount": 100, "category": "food"}},
        {{"amount": 500, "category": "travel"}}
    ]
    """
    
    response = llm.invoke(prompt).content
    
    try:
        # Convert the string output into an actual Python list
        return json.loads(response)
    except json.JSONDecodeError:
        return [{"error": "Failed to parse expenses."}]


@tool
def analyzer(expenses: list) -> dict:
    """Calculates totals using Python and generates insights using an LLM.""" 

    # 1. Do the deterministic math in Python!
    total_spent = sum(item.get("amount", 0) for item in expenses)
    
    category_breakdown = {}
    for item in expenses:
        cat = item.get("category", "other")
        category_breakdown[cat] = category_breakdown.get(cat, 0) + item.get("amount", 0)
        
    highest_category = max(category_breakdown, key=category_breakdown.get) if category_breakdown else "none"

    # 2. Ask the LLM ONLY for insights based on the accurate math
    prompt = f"""
    You are a personal finance assistant.
    
    Here is the accurate, calculated expense data for the user:
    - Total Spent: ${total_spent}
    - Category Breakdown: {category_breakdown}
    - Highest Category: {highest_category}
    
    Provide 2-3 brief insights or budgeting suggestions based on this data.
    Return ONLY a valid JSON list of strings. Do not include markdown formatting.
    
    Example:
    ["Insight 1 about food spending", "Insight 2 about saving"]
    """
    
    response = llm.invoke(prompt).content
    
    try:
        insights = json.loads(response)
    except json.JSONDecodeError:
        insights = ["Could not generate insights."]

    # 3. Combine the Python math and LLM insights into the final payload
    return {
        "total_spent": total_spent,
        "category_breakdown": category_breakdown,
        "highest_spending_category": highest_category,
        "insights": insights
    }

In [38]:
expense = input("Enter your input : ")
expenses_list = tracker.invoke({'expense':expense})
print(expenses_list)

Enter your input :  i refilled my car fuel tank for 2000/- and then had lunch with friends where my contribution was 500/- and 20/- for parking


[{'amount': 2000, 'category': 'fuel'}, {'amount': 500, 'category': 'food'}, {'amount': 20, 'category': 'parking'}]


In [46]:
analysis = analyzer.invoke({'expenses':expenses_list})
analysis

{'total_spent': 2520,
 'category_breakdown': {'fuel': 2000, 'food': 500, 'parking': 20},
 'highest_spending_category': 'fuel',
 'insights': ['Consider exploring fuel-efficient alternatives or routes to reduce the $2000 spent on fuel',
  ' Allocate a portion of the budget to savings to offset high fuel costs',
  'Review food spending to identify opportunities for reduction, as $500 is a significant secondary expense']}

In [48]:
for k,v in analysis.items():
    print(f"{k} : {v}")

total_spent : 2520
category_breakdown : {'fuel': 2000, 'food': 500, 'parking': 20}
highest_spending_category : fuel
insights : ['Consider exploring fuel-efficient alternatives or routes to reduce the $2000 spent on fuel', ' Allocate a portion of the budget to savings to offset high fuel costs', 'Review food spending to identify opportunities for reduction, as $500 is a significant secondary expense']
